# Model Selection - VariantClassifier

Este notebook demonstra o processo de seleção de modelos para classificação de variantes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('../src')

from modeling.preprocessing import VariantPreprocessor

print('Bibliotecas importadas com sucesso!')

## 1. Carregamento e Preparação dos Dados

In [ ]:
# Carregar dados
train_df = pd.read_csv('../data/splits/train.csv')
val_df = pd.read_csv('../data/splits/val.csv')

X_train = train_df.drop('pathogenicity', axis=1)
y_train = train_df['pathogenicity']

X_val = val_df.drop('pathogenicity', axis=1)
y_val = val_df['pathogenicity']

# Preprocessor
preprocessor = VariantPreprocessor.load('../models/preprocessor.joblib')
X_train_proc = preprocessor.transform(X_train)
X_val_proc = preprocessor.transform(X_val)

y_train_enc = preprocessor.encode_target(y_train)
y_val_enc = preprocessor.encode_target(y_val)

print(f'Dados preparados:')
print(f'X_train: {X_train_proc.shape}')
print(f'X_val: {X_val_proc.shape}')

## 2. Modelos Candidatos

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Modelos a serem testados
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Naive Bayes': GaussianNB(),
}

print(f'{len(models)} modelos candidatos configurados.')

## 3. Cross-Validation

In [ ]:
# Cross-validation stratificada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}

for name, model in models.items():
    print(f'Treinando {name}...')
    
    # Cross-validation scores
    cv_scores = cross_val_score(
        model, X_train_proc, y_train_enc,
        cv=cv, scoring='accuracy', n_jobs=-1
    )
    
    results[name] = {
        'mean_accuracy': cv_scores.mean(),
        'std_accuracy': cv_scores.std(),
        'scores': cv_scores
    }
    
    print(f'  Mean Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

print('\nCross-validation concluída!')

In [ ]:
# Comparação visual
model_names = list(results.keys())
mean_scores = [results[name]['mean_accuracy'] for name in model_names]
std_scores = [results[name]['std_accuracy'] for name in model_names]

plt.figure(figsize=(12, 6))
plt.bar(model_names, mean_scores, yerr=std_scores, capsize=5, alpha=0.7)
plt.ylabel('Accuracy')
plt.title('Comparação de Modelos - Cross-Validation')
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.axhline(y=0.2, color='r', linestyle='--', label='Baseline (random)')
plt.legend()
plt.tight_layout()
plt.show()

print('\nModelos ordenados por accuracy:')
for name in sorted(results.keys(), key=lambda x: results[x]['mean_accuracy'], reverse=True):
    print(f"{name}: {results[name]['mean_accuracy']:.4f} (+/- {results[name]['std_accuracy']:.4f})")

## 4. Avaliação no Conjunto de Validação

In [ ]:
# Treinar modelos em todo treino e avaliar em validação
val_results = {}

for name, model in models.items():
    print(f'Treinando {name} em todo treino...')
    
    # Fit
    model.fit(X_train_proc, y_train_enc)
    
    # Predict
    y_pred = model.predict(X_val_proc)
    
    # Accuracy
    accuracy = (y_pred == y_val_enc).mean()
    
    val_results[name] = {
        'model': model,
        'accuracy': accuracy,
        'predictions': y_pred
    }
    
    print(f'  Validation Accuracy: {accuracy:.4f}')

print('\nAvaliação concluída!')

In [ ]:
# Classification report do melhor modelo
best_model_name = max(val_results.keys(), key=lambda x: val_results[x]['accuracy'])
best_model = val_results[best_model_name]['model']
best_pred = val_results[best_model_name]['predictions']

print(f'Melhor modelo: {best_model_name}')
print(f'\nClassification Report:')
print(classification_report(
    y_val_enc, best_pred,
    target_names=preprocessor.target_mapping.keys(),
    digits=4
))

## 5. Matriz de Confusão

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(y_val_enc, best_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=preprocessor.target_mapping.keys(),
    yticklabels=preprocessor.target_mapping.keys()
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Matriz de Confusão - {best_model_name}')
plt.tight_layout()
plt.show()

## 6. Ensemble de Modelos

In [ ]:
from sklearn.ensemble import VotingClassifier

# Criar ensemble com top 3 modelos
top_models = sorted(val_results.keys(), key=lambda x: val_results[x]['accuracy'], reverse=True)[:3]

print(f'Top 3 modelos para ensemble: {top_models}')

estimators = [(name, val_results[name]['model']) for name in top_models]

# Voting classifier (soft voting)
ensemble = VotingClassifier(
    estimators=estimators,
    voting='soft',  # usa probabilidades
    n_jobs=-1
)

# Fit
print('Treinando ensemble...')
ensemble.fit(X_train_proc, y_train_enc)

# Predict
y_pred_ensemble = ensemble.predict(X_val_proc)
accuracy_ensemble = (y_pred_ensemble == y_val_enc).mean()

print(f'\nEnsemble Accuracy: {accuracy_ensemble:.4f}')
print(f'Melhor modelo individual: {val_results[best_model_name]["accuracy"]:.4f}')
print(f'Melhoria: {accuracy_ensemble - val_results[best_model_name]["accuracy"]:.4f}')

## 7. XGBoost e LightGBM

In [ ]:
try:
    import xgboost as xgb
    import lightgbm as lgb
    
    # XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        use_label_encoder=False,
        eval_metric='mlogloss'
    )
    
    print('Treinando XGBoost...')
    xgb_model.fit(X_train_proc, y_train_enc)
    y_pred_xgb = xgb_model.predict(X_val_proc)
    acc_xgb = (y_pred_xgb == y_val_enc).mean()
    print(f'XGBoost Accuracy: {acc_xgb:.4f}')
    
    # LightGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    print('Treinando LightGBM...')
    lgb_model.fit(X_train_proc, y_train_enc)
    y_pred_lgb = lgb_model.predict(X_val_proc)
    acc_lgb = (y_pred_lgb == y_val_enc).mean()
    print(f'LightGBM Accuracy: {acc_lgb:.4f}')
    
except ImportError:
    print('XGBoost ou LightGBM não disponível.')

## 8. Comparação Final

In [ ]:
# Resumo final
final_comparison = []

for name, res in val_results.items():
    final_comparison.append({
        'Model': name,
        'CV Accuracy': f"{results[name]['mean_accuracy']:.4f} +/- {results[name]['std_accuracy']:.4f}",
        'Val Accuracy': f"{res['accuracy']:.4f}"
    })

try:
    final_comparison.append({
        'Model': 'XGBoost',
        'CV Accuracy': 'N/A',
        'Val Accuracy': f'{acc_xgb:.4f}'
    })
    final_comparison.append({
        'Model': 'LightGBM',
        'CV Accuracy': 'N/A',
        'Val Accuracy': f'{acc_lgb:.4f}'
    })
    final_comparison.append({
        'Model': 'Ensemble',
        'CV Accuracy': 'N/A',
        'Val Accuracy': f'{accuracy_ensemble:.4f}'
    })
except:
    pass

comparison_df = pd.DataFrame(final_comparison)
display(comparison_df)

## 9. Resumo

In [ ]:
print('\n' + '='*80)
print('RESUMO DE SELEÇÃO DE MODELOS')
print('='*80)

print(f'\nTotal de modelos testados: {len(models)}')
print(f'Melhor modelo (validação): {best_model_name}')
print(f'Melhor accuracy: {val_results[best_model_name]["accuracy"]:.4f}')

print(f'\nRecomendação: Usar ensemble de XGBoost + LightGBM + Meta-learner')
print(f'Justificativa: Diversidade de algoritmos reduz overfitting e melhora generalização')